# Exercise 2: Modify History Frequency


<div class="alert alert-info">

## Exercise: Create a daily history file

Create a new case named `i2000_bgc_hist_daily` using the compset `I2000Clm60BgcCrujra` and resolution `f19_f19_mt233`.

By default CLM writes one monthly history file. Add a second history file that writes daily averages of ``TSA`` (2-m air temperature) and ``TLAI`` (leaf area index).

</div>



<div class="alert alert-warning">  
<details>

<summary> <font face="Times New Roman" color='blue'>Click here for hints</font> </summary>
    
**How do I control the output?**

Use CLM namelist variables:

```fortran
hist_fincl1 and hist_fincl2
hist_nhtfrq
hist_mfilt
```

Remember that if `hist_nhtfrq = 0`, the file contains monthly averages, if `hist_nhtfrq > 0`, the output frequency is given as a number of model time steps, and if `hist_nhtfrq < 0`, the output frequency is given as a number of hours.

**Think about:** Each value in ``hist_nhtfrq`` and ``hist_mfilt`` lines up with a history file (the first is for h0, the second for h1). A negative ``hist_nhtfrq`` is interpreted as hours, and ``0`` means monthly. What does ``hist_mfilt`` control? 

If you left ``hist_mfilt`` out and the daily file never appeared after a 1-month run, why might that be?

</details>
</div>




<div class="alert alert-success">   
<details>
<summary><font face="Times New Roman" color='blue'>Click here for the solution</font></summary><br>
                
**Create a new case**

Create a new case `i2000_bgc_hist_daily` with the command:

```bash
cd /glade/u/home/$USER/code/my_clm_code/cime/scripts/

./create_newcase --case ~/cases/i2000_bgc_hist_daily \
  --compset I2000Clm60BgcCrujra \
  --res f19_f19_mt233 \
  --run-unsupported
```

**Setup**
    
Invoke `case.setup` with the command:

```bash
cd ~/cases/i2000_bgc_hist_daily
./case.setup
```

**Customize namelists**
    
Edit the file `user_nl_clm` and add the lines:

```fortran
hist_fincl2 = 'TSA','TLAI'
hist_nhtfrq = 0,-24
hist_mfilt  = 1,31
```
 
**Set run length**
    
Change the run length:

```bash
./xmlchange STOP_N=1
./xmlchange STOP_OPTION=nmonths
```

**Change the wallclock time, job queue, and account number**

Change the wallclock time. If needed, change `job queue` and `account number`. <br>
For instance, to run in the queue `tutorial` and the project number ``UESM0016`` (You should use the project number given for this tutorial), use the command:

```
./xmlchange JOB_WALLCLOCK_TIME=01:00:00
./xmlchange JOB_QUEUE=tutorial
./xmlchange PROJECT=UESM0016 --force
```

**Build and submit**
    
Build the model and submit your job:
```
qcmd -- ./case.build
./case.submit
```
____
    
**Look at your solution**

When the run is complete, go to the archive directory:

```bash
cd /glade/derecho/scratch/$USER/archive/i2000_bgc_hist_daily/lnd/hist
ls
```

In CLM, history files are separated into averaged and instantaneous files using the suffixes `a` and `i`.

You should see files like:

```text
- h0 
i2000_bgc_hist_daily.clm2.h0a.2000-01.nc
i2000_bgc_hist_daily.clm2.h0i.2000-01.nc

- h1
i2000_bgc_hist_daily.clm2.h1a.2000-01-02-00000.nc
```

The important files for this exercise are:

* `h0a`: monthly averaged output
* `h1a`: daily averaged output requested with `fincl2 = 'TSA', 'TLAI'`

For this exercise, focus on checking:

```bash
ncdump -h i2000_bgc_hist_daily.clm2.h1a.2000-01-02-00000.nc
```

The `h1a` files should contain the daily time-averaged fields. These variables should include:

```text
double time_bounds(time, nbnd) ;
        time_bounds:long_name = "time interval endpoints" ;
        time_bounds:units = "days since 2000-01-01 00:00:00" ;
        time_bounds:calendar = "noleap" ;

float TLAI(time, lat, lon) ;
          TLAI:long_name = "total projected leaf area index" ;
          TLAI:units = "m^2/m^2" ;
          TLAI:cell_methods = "time: mean" ;
          TLAI:_FillValue = 1.e+36f ;
          TLAI:missing_value = 1.e+36f ;
          TLAI:landunit_mask = "unknown" ;
          
float TSA(time, lat, lon) ;
        TSA:long_name = "2m air temperature" ;
        TSA:units = "K" ;
        TSA:cell_methods = "time: mean" ;
        TSA:_FillValue = 1.e+36f ;
        TSA:missing_value = 1.e+36f ;
        TSA:landunit_mask = "unknown" ;
```

Check the number of time samples:

```bash
ncdump -h i2000_bgc_hist_daily.clm2.h1a.2000-01-02-00000.nc | grep time
```

The `h1a` file should contain 31 time samples.

You can also check file sizes with:

```bash
du -skh /glade/derecho/scratch/$USER/archive/i2000_bgc_hist_daily/lnd/hist/*
```

</details>
</div>

